# Bronze　Layer

## 目的・方針

- `workspace.practice`（ソースシステム相当）の生データを、無加工に近い形で `workspace.bronze` に取り込む
- 対象テーブル: `products`（商品マスタ）, `sales`（売上）, `inventory`（在庫）
- Bronze層では以下のメタデータ列のみ付与し、カラムの加工・型変換・フィルタは行わない
    - `_ingested_at`: 取り込み時刻
    - `_source_table`: 取り込み元テーブル
- ソース側が毎回フルスナップショットで再生成される想定のため、書き込みは `overwrite` とする（再実行しても結果が変わらない = 冪等）


In [0]:
from pyspark.sql.functions import lit, col, current_timestamp

SOURCE_FILE_PATHS = [
    "/Volumes/workspace/practice/vol/products.csv",
    "/Volumes/workspace/practice/vol/sales.csv",
    "/Volumes/workspace/practice/vol/iventory.csv",
]


BRONZE_SCHEMA = "workspace.bronze"
BRONZE_TABLES = ["products", "sales", "inventory"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

In [0]:
def ingest_to_bronze(file_path: str, table_name: str) -> None:

    """
    ソーステーブルをメタデータ付与のみでBronzeテーブルへ取り込む

    bronzeテーブル _10_の接頭辞を付与
    """

    target_table = f"{BRONZE_SCHEMA}._10_bronze_{table_name}"

    df_source = spark.read.csv(file_path, header=True)
    df_bronze = df_source.withColumn("_ingested_at", current_timestamp()).withColumn(
        "_source_table", lit(file_path)
    )

    (
        df_bronze.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Bronze取り込み完了: {target_table}")

In [0]:
for i in range(3):
    ingest_to_bronze(SOURCE_FILE_PATHS[i], BRONZE_TABLES[i])

In [0]:
# 取り込み結果の確認
for table_name in BRONZE_TABLES:
    df = spark.read.table(f"{BRONZE_SCHEMA}._10_bronze_{table_name}")
    print(f"{table_name}: {df.count()}件")
    display(df)